# Notebook 6 — Interactive dashboard

Four panels: leaderboard, per-artist momentum (dropdown), ROC + confusion matrix, novel predictions. Requires `scikit-learn` for ROC if not installed: `pip install scikit-learn --user`.


In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("."))
import hdfs_paths as hp

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession.builder.appName("MusicTrend_06_Dashboard")
    .config("spark.sql.shuffle.partitions", hp.SHUFFLE_PARTITIONS)
    .getOrCreate()
)

FEATURE_COLS = [
    "plays_1d",
    "plays_7d",
    "plays_28d",
    "growth_rate_7d",
    "stream_velocity",
    "region_spread",
    "total_streams",
    "tempo",
    "energy",
    "loudness",
    "danceability",
]
LABEL_COL = "charted"

features = spark.read.parquet(hp.PROCESSED_FEATURES)
try:
    stream_out = spark.read.json(hp.STREAMING_OUTPUT)
    stream_out.createOrReplaceTempView("stream_out")
except Exception as e:
    print("Streaming output optional:", e)

from pyspark.ml import PipelineModel

model = PipelineModel.load(hp.MODEL_RF)
predictions = model.transform(features.filter(col("week_year") >= 2021))


In [ ]:
# Panel 1 — leaderboard by peak growth_rate_7d in latest (week_year, week_number)
import matplotlib.pyplot as plt
from pyspark.sql.functions import struct, max as spark_max

mx = features.select(spark_max(struct("week_year", "week_number")).alias("m")).first()[0]
recent_spark = features.filter(
    (col("week_year") == mx.week_year) & (col("week_number") == mx.week_number)
)
leaderboard_df = recent_spark.orderBy(col("growth_rate_7d").desc()).limit(20).toPandas()

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#2ca02c" if c else "#7f7f7f" for c in leaderboard_df["charted"]]
ax.barh(leaderboard_df["artist_norm"][::-1], leaderboard_df["growth_rate_7d"][::-1], color=colors[::-1])
ax.set_title("Top 20 by growth_rate_7d (latest week in data)")
plt.tight_layout()
plt.show()


In [ ]:
# Panel 2 — dropdown trendline
import ipywidgets as widgets
from IPython.display import display

features_pd = features.toPandas()
top50 = (
    features_pd.sort_values("plays_7d", ascending=False)["artist_norm"].drop_duplicates().head(50).tolist()
)
dropdown = widgets.Dropdown(options=top50, description="Artist:")


def plot_artist(artist):
    data = (
        features_pd[features_pd["artist_norm"] == artist]
        .sort_values(["week_year", "week_number"])
        .reset_index(drop=True)
    )
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(data.index, data["plays_7d"], label="7-day plays")
    hit = data[data["charted"] == 1]
    if not hit.empty:
        ax.axvline(x=float(hit.index[0]), color="red", linestyle="--", label="Chart-labeled week")
    ax.set_title(f"Momentum: {artist}")
    ax.legend()
    plt.show()


widgets.interactive(plot_artist, artist=dropdown)


In [ ]:
# Panel 3 — ROC + confusion matrix heatmap
from sklearn.metrics import roc_curve, auc as sk_auc
import numpy as np

preds_pd = predictions.select(LABEL_COL, "probability", "prediction").toPandas()
preds_pd["prob_1"] = preds_pd["probability"].apply(lambda v: float(v[1]))

fpr, tpr, _ = roc_curve(preds_pd[LABEL_COL], preds_pd["prob_1"])
roc_auc = sk_auc(fpr, tpr)

cm_df = predictions.groupBy(LABEL_COL, "prediction").count().toPandas()
pivot = cm_df.pivot(index=LABEL_COL, columns="prediction", values="count").fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
axes[0].plot([0, 1], [0, 1], "k--")
axes[0].set_title("ROC Curve")
axes[0].legend()

im = axes[1].imshow(pivot.values, cmap="Blues")
axes[1].set_xticks(range(pivot.shape[1]))
axes[1].set_yticks(range(pivot.shape[0]))
axes[1].set_xticklabels(pivot.columns)
axes[1].set_yticklabels(pivot.index)
axes[1].set_title("Confusion matrix")
for (i, j), val in np.ndenumerate(pivot.values):
    axes[1].text(j, i, int(val), ha="center", va="center", color="black")
plt.tight_layout()
plt.show()


In [ ]:
# Panel 4 — novel predictions (predict 1, label 0)
novel = (
    predictions.filter((col("prediction") == 1) & (col(LABEL_COL) == 0))
    .select(
        "artist_norm",
        "week_year",
        "week_number",
        "probability",
        "growth_rate_7d",
    )
    .orderBy(col("probability").desc())
)
novel.show(20, truncate=False)


## Outputs confirmed

All four panels executed; model + features + streaming (if present) consumed from configured project paths.
